# Training

In [ ]:
import sys
import os
import time
import torch
import torch.nn as nn
from torch.optim import Adam

sys.path.append(os.path.abspath('..'))
from src.model import HybridGraphSAGE

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = HybridGraphSAGE(in_channels=128, hidden_channels=64, out_channels=2).to(device)
criterion = nn.CrossEntropyLoss()

learning_rate = 0.001 
epochs = 10 
optimizer = Adam(model.parameters(), lr=learning_rate) 

print(f"Starting training on {device}...")
model.train()

history = {'loss': [], 'accuracy': [], 'time': []}

for epoch in range(epochs):
    start_time = time.time()
    total_loss = 0
    correct = 0
    total_samples = 0
    
    for batch in full_loader: 
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        
        preds = out.argmax(dim=1)
        correct += int((preds == batch.y).sum())
        total_samples += batch.num_graphs
        
    epoch_duration = time.time() - start_time
    epoch_acc = correct / total_samples
    avg_loss = total_loss / total_samples
    
    history['loss'].append(avg_loss)
    history['accuracy'].append(epoch_acc)
    history['time'].append(epoch_duration)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {epoch_acc:.4f} | Time: {epoch_duration:.2f}s")

os.makedirs('../models', exist_ok=True)
model_save_path = '../models/hybrid_graphsage_weights.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\nTraining complete. Model weights saved to {model_save_path}")